In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings('ignore')

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
print(train.shape, test.shape)

(2000, 8) (500, 7)


In [5]:
def apk(actual, predicted, k=3):
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p == actual:
            hits += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, preds):
    return float(np.mean([apk(a, p) for a, p in zip(actuals, preds)]))

In [6]:
rows = []
for _, r in train.iterrows():
    for c in OPTION_COLS:
        rows.append({
            'text':   str(r['prompt']) + ' ' + str(r[c]),
            'qid':    r['id'],
            'option': c,
            'label':  1 if r['answer'] == c else 0
        })
pw_train = pd.DataFrame(rows)

rows = []
for _, r in test.iterrows():
    for c in OPTION_COLS:
        rows.append({
            'text':   str(r['prompt']) + ' ' + str(r[c]),
            'qid':    r['id'],
            'option': c
        })
pw_test = pd.DataFrame(rows)

print(pw_train.shape, pw_test.shape)

(10000, 4) (2500, 3)


In [7]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
X_tr  = tfidf.fit_transform(pw_train['text'])
X_tst = tfidf.transform(pw_test['text'])
y_tr  = pw_train['label'].values
groups = pw_train['qid'].values

In [8]:
def rank_preds(pw, scores, orig_df):
    pw = pw.copy()
    pw['score'] = scores
    return [
        pw[pw['qid'] == qid].sort_values('score', ascending=False)['option'].tolist()[:3]
        for qid in orig_df['id']
    ]

gkf    = GroupKFold(n_splits=5)
oof    = np.zeros(len(pw_train))
tst_lr = np.zeros(len(pw_test))

for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, groups)):
    lr = LogisticRegression(C=4.0, max_iter=1000, n_jobs=-1)
    lr.fit(X_tr[ti], y_tr[ti])
    oof[vi] = lr.predict_proba(X_tr[vi])[:, 1]
    tst_lr += lr.predict_proba(X_tst)[:, 1] / 5

    val_qids  = pw_train.iloc[vi]['qid'].unique()
    val_train = train[train['id'].isin(val_qids)]
    val_pw    = pw_train[pw_train['qid'].isin(val_qids)]
    val_scores= oof[pw_train['qid'].isin(val_qids).values]
    fold_map3 = mapk(val_train['answer'].tolist(), rank_preds(val_pw, val_scores, val_train))
    print(f"Fold {fold+1}  MAP@3={fold_map3:.4f}")

oof_map3 = mapk(train['answer'].tolist(), rank_preds(pw_train, oof, train))
print(f"\nOOF MAP@3: {oof_map3:.4f}")

Fold 1  MAP@3=0.9925
Fold 2  MAP@3=0.9800
Fold 3  MAP@3=0.9850
Fold 4  MAP@3=0.9900
Fold 5  MAP@3=0.9875

OOF MAP@3: 0.9870


In [ ]:
test_preds = rank_preds(pw_test, tst_lr, test)

submission = pd.DataFrame({
    'ID':         test['id'],
    'Prediction': [' '.join(p) for p in test_preds]
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10).to_string(index=False))